# F, G, H. 평가 종합 · 보고서 생성 · 보고서 검증

| | F. 평가 종합 | G. 보고서 생성 | H. 보고서 검증 |
|---|---|---|---|
| **담당** | LLM (검색 없음) | LLM (검색 없음) | 규칙 기반 (LLM 없음) |
| **선행 노드** | C, D, E | F (또는 H 재진입) | G |
| **출력** | `synthesis` | `final_report` | `validation_result`, `retry_count` |

셋이 순서대로 이어지고(F→G→H) H가 실패하면 G로 되돌아가는 유일한 루프라 한 노트북에 같이 둔다.
H는 LLM을 안 쓰는 규칙 기반 노드라는 게 포인트 — 챕터 존재 여부·서열 표현 포함 여부를 문자열 검사로만 판단한다.

이 노트북 끝에서 만든 것들은 `src/nodes_fgh.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import config, prompts
from src.schemas import Synthesis, ValidationResult

## 1. F. 평가 종합 — 프롬프트 확인

In [ ]:
print(prompts.SYNTHESIS_PROMPT)

## 2. F. 노드 함수 정의

`market_eval`/`stakeholder_eval`/`domain_eval`은 C/D/E가 각자 쓴 State 키에서 읽고,
TRL은 `tech_research`(B가 씀) 안에 있어서 여기서 따로 뽑아낸다 — TRL 전담 에이전트가 없기 때문(2-1절).

In [ ]:
def make_node_f(llm):
    """F. 평가 종합. 4관점(시장/이해관계자/도메인/TRL) 라벨을 모아 비교한다."""
    structured_llm = llm.with_structured_output(Synthesis)

    def node_f_synthesis(state):
        tech_research = state.get("tech_research", {})
        trl_eval = {
            name: r.get("trl_assessment", {}) for name, r in tech_research.items()
        }
        prompt = prompts.SYNTHESIS_PROMPT.format(
            market_eval=state.get("market_eval", {}),
            stakeholder_eval=state.get("stakeholder_eval", {}),
            domain_eval=state.get("domain_eval", {}),
            trl_eval=trl_eval,
        )
        result = structured_llm.invoke(prompt)
        return {"synthesis": result.model_dump()}

    return node_f_synthesis

## 3. G. 보고서 생성 — 프롬프트 확인

4개로 나뉜 `*_references`를 여기서 하나로 합친다(reducer가 아니라 그냥 리스트 덧셈).
분석 배경은 State가 아니라 `config.ANALYSIS_BACKGROUND` 고정 문단을 그대로 쓴다.

In [ ]:
print(prompts.REPORT_PROMPT)

In [ ]:
def make_node_g(llm):
    """G. 보고서 생성. H가 무효 판정을 내리면 재진입해서 revision_note를 받는다."""
    def node_g_report(state):
        validation = state.get("validation_result")
        revision_note = ""
        if validation and not validation.get("is_valid", True):
            missing = ", ".join(validation.get("missing_items", []))
            revision_note = (
                f"[재작성 지시] 이전 초안에서 다음이 빠졌다: {missing}. "
                "이번엔 반드시 포함하라."
            )

        all_references = (
            state.get("tech_references", [])
            + state.get("market_references", [])
            + state.get("stakeholder_references", [])
            + state.get("domain_references", [])
        )

        prompt = prompts.REPORT_PROMPT.format(
            analysis_background=config.ANALYSIS_BACKGROUND,
            selected_technologies=state.get("selected_technologies", {}),
            tech_research=state.get("tech_research", {}),
            synthesis=state.get("synthesis", {}),
            references=all_references,
            revision_note=revision_note,
        )
        report_text = llm.invoke(prompt).content
        return {"final_report": report_text}

    return node_g_report

## 4. H. 보고서 검증 — 규칙 정의

LLM을 안 부른다. `REQUIRED_CHAPTERS`가 전부 본문에 있는지, `FORBIDDEN_WORDS`(서열 표현)가 안 섞였는지만 문자열로 검사한다.
`retry_count`가 `config.MAX_RETRY_H`(2)를 넘으면 강제로 통과시키되 `forced_pass=True`로 표시한다 — G가 이걸 보고 「한계점」 장에 뭐가 빠졌는지 적는다.

In [ ]:
REQUIRED_CHAPTERS = ["SUMMARY", "시장", "이해관계자", "도메인", "REFERENCE"]
FORBIDDEN_WORDS = ["우수", "우월", "우위", "권장", "추천"]


def node_h_validate(state):
    report = state.get("final_report", "")
    retry_count = state.get("retry_count", 0)

    missing = [c for c in REQUIRED_CHAPTERS if c not in report]
    forbidden_found = [w for w in FORBIDDEN_WORDS if w in report]
    if forbidden_found:
        missing.append(f"서열 표현 발견: {forbidden_found}")

    if not missing:
        result = ValidationResult(is_valid=True, missing_items=[], forced_pass=False)
        return {"validation_result": result.model_dump(), "retry_count": retry_count}

    new_retry_count = retry_count + 1
    if new_retry_count > config.MAX_RETRY_H:
        result = ValidationResult(is_valid=True, missing_items=missing, forced_pass=True)
    else:
        result = ValidationResult(is_valid=False, missing_items=missing, forced_pass=False)

    return {"validation_result": result.model_dump(), "retry_count": new_retry_count}


def route_after_h(state):
    """H 다음 조건부 엣지. graph.py의 add_conditional_edges가 이 함수를 쓴다."""
    validation = state.get("validation_result", {})
    if validation.get("is_valid", False):
        return "END"
    return "G"

## 5. 배선 테스트 — API 키 없이

H의 재시도 루프(최초 1회 + 재시도 2회 = 총 3회, 상한 도달 시 강제통과)가 제대로 도는지까지 확인한다.

In [ ]:
class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeMessage:
    def __init__(self, content):
        self.content = content

class FakeLLM_F:
    def with_structured_output(self, schema_cls):
        return FakeStructuredLLM(Synthesis(
            labels={"market": "조건 의존", "stakeholder": "조건 의존", "domain": "조건 의존", "trl": "판단보류"},
            conflicts=[], reasoning="가짜 이유",
        ))

# F 테스트
node_f = make_node_f(FakeLLM_F())
f_result = node_f({})
assert "conflicts" in f_result["synthesis"]
print("F 배선 OK")

# G, H 테스트: G가 매번 다른 보고서를 내도록 만들어서 H->G 재시도 루프를 직접 확인
call_count = {"n": 0}
class FakeLLM_G:
    def invoke(self, prompt):
        call_count["n"] += 1
        if call_count["n"] == 1:
            return FakeMessage("# SUMMARY\n내용\n# 시장\n내용\n# 이해관계자\n내용\n# REFERENCE\n내용")  # 도메인 누락
        return FakeMessage("# SUMMARY\n내용\n# 시장\n내용\n# 이해관계자\n내용\n# 도메인\n내용\n# REFERENCE\n내용")

node_g = make_node_g(FakeLLM_G())

state = {}
state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "G"  # 도메인 누락이라 재시도
print("1차 검증:", state["validation_result"])

state.update(node_g(state))
state.update(node_h_validate(state))
assert route_after_h(state) == "END"  # 이번엔 통과
print("2차 검증:", state["validation_result"])
print("G/H 배선 OK, retry_count =", state["retry_count"])

## 6. 실제 LLM 테스트 (F만 — G/H는 최종 통합 노트북에서 실제 데이터로 보는 게 더 의미 있다)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY"):
    from langchain.chat_models import init_chat_model

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    node_f_real = make_node_f(real_llm)
    sample_state = {
        "tech_research": {"TurboQuant": {"trl_assessment": {"trl_ondevice": 4}}, "InfiniGen": {"trl_assessment": {"trl_ondevice": 3}}},
        "market_eval": {"label": "조건 의존"},
        "stakeholder_eval": {"label": "조건 의존"},
        "domain_eval": {"label": "조건 의존"},
    }
    print(node_f_real(sample_state)["synthesis"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 7. 파일로 저장

In [ ]:
import inspect

q3 = chr(34) * 3
with open("../src/nodes_fgh.py", "w", encoding="utf-8") as f:
    f.write(q3 + "F, G, H. 평가종합/보고서생성/보고서검증 노드 - 04_agent_FGH_synthesis_report_validate.ipynb에서 생성됨.\n")
    f.write("이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n")
    f.write("from src import config, prompts\n")
    f.write("from src.schemas import Synthesis, ValidationResult\n\n\n")
    f.write(inspect.getsource(make_node_f))
    f.write("\n\n")
    f.write(inspect.getsource(make_node_g))
    f.write("\n\n")
    f.write(f'REQUIRED_CHAPTERS = {REQUIRED_CHAPTERS!r}\n')
    f.write(f'FORBIDDEN_WORDS = {FORBIDDEN_WORDS!r}\n\n\n')
    f.write(inspect.getsource(node_h_validate))
    f.write("\n\n")
    f.write(inspect.getsource(route_after_h))
    f.write("\n")

print("src/nodes_fgh.py 저장 완료")